# Ranking Window Functions

## Overview
Ranking functions assign a position number to each row within a partition based on an ORDER BY clause. They are among the most commonly used window functions in practice — top-N per group, percentile bucketing, deduplication.

| Function | Tie handling | Gap after tie |
|---|---|---|
| `ROW_NUMBER()` | Assigns distinct numbers — ties get arbitrary order | No — always 1,2,3,4... |
| `RANK()` | Tied rows get the same rank | Yes — next rank skips (1,1,3) |
| `DENSE_RANK()` | Tied rows get the same rank | No — next rank is consecutive (1,1,2) |
| `NTILE(n)` | Divides rows into n roughly equal buckets | N/A |
| `PERCENT_RANK()` | Relative rank as 0.0–1.0 | N/A — continuous |
| `CUME_DIST()` | Cumulative distribution — fraction of rows ≤ current | N/A — continuous |

**All ranking functions require an ORDER BY inside OVER.**

**Top-N per group pattern** (the most common interview question):
```sql
SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY group_col ORDER BY value_col DESC) AS rn
    FROM table
) WHERE rn <= N;
```

---

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, segment TEXT, province TEXT);
CREATE TABLE accounts (account_id INTEGER PRIMARY KEY, customer_id INTEGER, account_type TEXT, province TEXT, opened_date TEXT, status TEXT);
CREATE TABLE transactions (txn_id INTEGER PRIMARY KEY, account_id INTEGER, txn_date TEXT, txn_type TEXT, amount REAL, category TEXT, flagged INTEGER);
CREATE TABLE trades (trade_id INTEGER PRIMARY KEY, account_id INTEGER, trade_date TEXT, ticker TEXT, direction TEXT, shares INTEGER, price REAL, total_value REAL);
INSERT INTO customers VALUES
  (1,'Aroha','Ngata','Retail','NB'),(2,'Liam','Chen','SME','BC'),
  (3,'Fatima','Al-Rashid','Wealth','ON'),(4,'James','MacLeod','Retail','NB'),
  (5,'Sofia','Petrov','SME','BC'),(6,'Noah','Williams','Retail','AB'),
  (7,'Mei','Zhang','Wealth','ON'),(8,'Ethan','Tremblay','Retail','QC'),
  (9,'Priya','Nair','SME','BC'),(10,'Marcus','Okafor','Retail','ON');
INSERT INTO accounts VALUES
  (101,1,'Chequing','NB','2019-03-15','Active'),(102,1,'Savings','NB','2019-03-15','Active'),
  (103,2,'Chequing','BC','2020-07-01','Active'),(104,3,'Investment','ON','2018-05-20','Active'),
  (105,4,'Chequing','NB','2015-09-30','Active'),(106,5,'Chequing','BC','2021-06-15','Active'),
  (107,6,'Chequing','AB','2017-11-22','Active'),(108,7,'Investment','ON','2016-03-08','Active'),
  (109,8,'Chequing','QC','2023-01-05','Active'),(110,9,'Investment','BC','2022-08-19','Active'),
  (111,10,'Chequing','ON','2020-12-01','Active');
INSERT INTO transactions VALUES
  (1001,101,'2023-01-05','Deposit',4200.00,'Payroll',0),(1002,101,'2023-01-12','Withdrawal',-850.00,'Rent',0),
  (1003,101,'2023-01-20','Withdrawal',-120.00,'Groceries',0),(1004,101,'2023-02-05','Deposit',4200.00,'Payroll',0),
  (1005,101,'2023-02-18','Withdrawal',-980.00,'Rent',0),(1006,101,'2023-03-05','Deposit',4200.00,'Payroll',0),
  (1007,103,'2023-01-08','Deposit',15000.00,'Payroll',0),(1008,103,'2023-01-25','Withdrawal',-3200.00,'Rent',0),
  (1009,103,'2023-02-08','Deposit',15000.00,'Payroll',0),(1010,103,'2023-02-22','Withdrawal',-2800.00,'Rent',0),
  (1011,103,'2023-03-08','Deposit',15000.00,'Payroll',0),(1012,105,'2023-01-06','Deposit',3100.00,'Payroll',0),
  (1013,105,'2023-01-19','Withdrawal',-700.00,'Rent',0),(1014,105,'2023-02-06','Deposit',3100.00,'Payroll',0),
  (1015,105,'2023-02-20','Withdrawal',-650.00,'Groceries',0),(1016,107,'2023-01-20','Deposit',3800.00,'Payroll',0),
  (1017,107,'2023-02-10','Fee',-25.00,'Fee',1),(1018,107,'2023-03-15','Withdrawal',-450.00,'Groceries',1),
  (1019,109,'2023-01-10','Deposit',2800.00,'Payroll',0),(1020,109,'2023-01-28','Withdrawal',-650.00,'Groceries',0),
  (1021,111,'2023-01-22','Deposit',4500.00,'Payroll',0),(1022,111,'2023-02-15','Withdrawal',-1100.00,'Utilities',0),
  (1023,111,'2023-03-01','Deposit',4500.00,'Payroll',0);
INSERT INTO trades VALUES
  (2001,104,'2023-01-10','AAPL','Buy',10,148.50,1485.00),(2002,104,'2023-01-25','MSFT','Buy',5,240.10,1200.50),
  (2003,104,'2023-02-14','AAPL','Buy',5,153.20,766.00),(2004,104,'2023-02-28','MSFT','Sell',3,252.80,758.40),
  (2005,104,'2023-03-15','NVDA','Buy',2,278.50,557.00),(2006,108,'2023-01-05','AAPL','Buy',20,130.50,2610.00),
  (2007,108,'2023-01-18','AMZN','Buy',8,95.20,761.60),(2008,108,'2023-02-08','AAPL','Sell',10,151.00,1510.00),
  (2009,108,'2023-02-22','AMZN','Buy',5,99.80,499.00),(2010,108,'2023-03-10','NVDA','Buy',4,265.30,1061.20),
  (2011,110,'2023-01-12','MSFT','Buy',8,238.00,1904.00),(2012,110,'2023-02-01','AAPL','Buy',15,145.00,2175.00),
  (2013,110,'2023-02-20','MSFT','Buy',3,248.50,745.50),(2014,110,'2023-03-05','AAPL','Sell',10,155.00,1550.00),
  (2015,110,'2023-03-20','NVDA','Buy',6,280.00,1680.00);
""")
conn.commit()
print('Finance schema ready.')


Finance schema ready.


---
## ROW_NUMBER, RANK, DENSE_RANK — side by side

In [2]:
# Show the three ranking functions together with a tie scenario
# Trades by account, sorted by total_value descending — some tickers have same-day buys
print('=== ROW_NUMBER vs RANK vs DENSE_RANK on trade values ===')
q = """
SELECT  trade_id,
        account_id,
        trade_date,
        ticker,
        total_value,
        ROW_NUMBER()  OVER (PARTITION BY account_id ORDER BY total_value DESC) AS row_num,
        RANK()        OVER (PARTITION BY account_id ORDER BY total_value DESC) AS rnk,
        DENSE_RANK()  OVER (PARTITION BY account_id ORDER BY total_value DESC) AS dense_rnk
FROM    trades
ORDER BY account_id, total_value DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('Note: ROW_NUMBER always produces unique numbers within the partition.')
print('RANK skips numbers after a tie. DENSE_RANK does not skip.')


=== ROW_NUMBER vs RANK vs DENSE_RANK on trade values ===
 trade_id  account_id trade_date ticker  total_value  row_num  rnk  dense_rnk
     2001         104 2023-01-10   AAPL       1485.0        1    1          1
     2002         104 2023-01-25   MSFT       1200.5        2    2          2
     2003         104 2023-02-14   AAPL        766.0        3    3          3
     2004         104 2023-02-28   MSFT        758.4        4    4          4
     2005         104 2023-03-15   NVDA        557.0        5    5          5
     2006         108 2023-01-05   AAPL       2610.0        1    1          1
     2008         108 2023-02-08   AAPL       1510.0        2    2          2
     2010         108 2023-03-10   NVDA       1061.2        3    3          3
     2007         108 2023-01-18   AMZN        761.6        4    4          4
     2009         108 2023-02-22   AMZN        499.0        5    5          5
     2012         110 2023-02-01   AAPL       2175.0        1    1          1
     20

---
## Top-N per group — the most common ranking pattern

In [3]:
# Top 2 transactions by absolute amount per account
print('=== Top 2 transactions by absolute amount per account ===')
q = """
SELECT *
FROM (
    SELECT  txn_id,
            account_id,
            txn_date,
            txn_type,
            amount,
            ROW_NUMBER() OVER (
                PARTITION BY account_id
                ORDER BY ABS(amount) DESC, txn_id   -- txn_id as tiebreaker
            ) AS rn
    FROM    transactions
)
WHERE rn <= 2
ORDER BY account_id, rn
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Most recent trade per ticker across all investment accounts
print()
print('=== Most recent trade per ticker (latest date per ticker) ===')
q2 = """
SELECT ticker, account_id, trade_date, direction, shares, price
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY ticker
               ORDER BY trade_date DESC, trade_id DESC
           ) AS rn
    FROM trades
)
WHERE rn = 1
ORDER BY ticker
"""
print(pd.read_sql(q2, conn).to_string(index=False))


=== Top 2 transactions by absolute amount per account ===
 txn_id  account_id   txn_date   txn_type  amount  rn
   1001         101 2023-01-05    Deposit  4200.0   1
   1004         101 2023-02-05    Deposit  4200.0   2
   1007         103 2023-01-08    Deposit 15000.0   1
   1009         103 2023-02-08    Deposit 15000.0   2
   1012         105 2023-01-06    Deposit  3100.0   1
   1014         105 2023-02-06    Deposit  3100.0   2
   1016         107 2023-01-20    Deposit  3800.0   1
   1018         107 2023-03-15 Withdrawal  -450.0   2
   1019         109 2023-01-10    Deposit  2800.0   1
   1020         109 2023-01-28 Withdrawal  -650.0   2
   1021         111 2023-01-22    Deposit  4500.0   1
   1023         111 2023-03-01    Deposit  4500.0   2

=== Most recent trade per ticker (latest date per ticker) ===
ticker  account_id trade_date direction  shares  price
  AAPL         110 2023-03-05      Sell      10  155.0
  AMZN         108 2023-02-22       Buy       5   99.8
  MSFT      

---
## NTILE — bucketing rows into equal groups

In [4]:
# Divide transactions into quartiles by absolute amount
print('=== Transactions bucketed into 4 quartiles by amount ===')
q = """
SELECT  txn_id,
        account_id,
        amount,
        NTILE(4) OVER (ORDER BY ABS(amount) ASC) AS quartile,
        CASE NTILE(4) OVER (ORDER BY ABS(amount) ASC)
            WHEN 1 THEN 'Q1 (lowest)'
            WHEN 2 THEN 'Q2'
            WHEN 3 THEN 'Q3'
            WHEN 4 THEN 'Q4 (highest)'
        END AS quartile_label
FROM    transactions
ORDER BY ABS(amount)
"""
print(pd.read_sql(q, conn).to_string(index=False))

# NTILE into deciles for loan principal distribution
print()
print('=== Trades by value split into tertiles per account ===')
q2 = """
SELECT  trade_id,
        account_id,
        ticker,
        total_value,
        NTILE(3) OVER (PARTITION BY account_id ORDER BY total_value) AS tertile
FROM    trades
ORDER BY account_id, total_value
"""
print(pd.read_sql(q2, conn).to_string(index=False))


=== Transactions bucketed into 4 quartiles by amount ===
 txn_id  account_id  amount  quartile quartile_label
   1017         107   -25.0         1    Q1 (lowest)
   1003         101  -120.0         1    Q1 (lowest)
   1018         107  -450.0         1    Q1 (lowest)
   1015         105  -650.0         1    Q1 (lowest)
   1020         109  -650.0         1    Q1 (lowest)
   1013         105  -700.0         1    Q1 (lowest)
   1002         101  -850.0         2             Q2
   1005         101  -980.0         2             Q2
   1022         111 -1100.0         2             Q2
   1010         103 -2800.0         2             Q2
   1019         109  2800.0         2             Q2
   1012         105  3100.0         2             Q2
   1014         105  3100.0         3             Q3
   1008         103 -3200.0         3             Q3
   1016         107  3800.0         3             Q3
   1001         101  4200.0         3             Q3
   1004         101  4200.0         3     

---
## PERCENT_RANK and CUME_DIST

In [5]:
# Relative ranking of each transaction within its account
print('=== PERCENT_RANK and CUME_DIST per account ===')
q = """
SELECT  txn_id,
        account_id,
        amount,
        ROUND(PERCENT_RANK() OVER (
            PARTITION BY account_id ORDER BY amount DESC), 3) AS pct_rank,
        ROUND(CUME_DIST()    OVER (
            PARTITION BY account_id ORDER BY amount DESC), 3) AS cume_dist
FROM    transactions
WHERE   account_id IN (101, 103)
ORDER BY account_id, amount DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('PERCENT_RANK: (rank - 1) / (total_rows - 1). First row = 0.0, last = 1.0')
print('CUME_DIST:    fraction of rows <= current value. First row > 0.0, last = 1.0')

# Practical use: flag transactions in top 10% by size
print()
print('=== Flag top-10% transactions by amount within each account ===')
q2 = """
SELECT txn_id, account_id, amount,
       ROUND(PERCENT_RANK() OVER (PARTITION BY account_id ORDER BY amount DESC), 3) AS pct_rank,
       CASE WHEN PERCENT_RANK() OVER (PARTITION BY account_id ORDER BY amount DESC) <= 0.10
            THEN 'Top 10%' ELSE 'Other' END AS size_tier
FROM transactions
ORDER BY account_id, amount DESC
"""
print(pd.read_sql(q2, conn).to_string(index=False))

conn.close()


=== PERCENT_RANK and CUME_DIST per account ===
 txn_id  account_id  amount  pct_rank  cume_dist
   1001         101  4200.0      0.00      0.500
   1004         101  4200.0      0.00      0.500
   1006         101  4200.0      0.00      0.500
   1003         101  -120.0      0.60      0.667
   1002         101  -850.0      0.80      0.833
   1005         101  -980.0      1.00      1.000
   1007         103 15000.0      0.00      0.600
   1009         103 15000.0      0.00      0.600
   1011         103 15000.0      0.00      0.600
   1010         103 -2800.0      0.75      0.800
   1008         103 -3200.0      1.00      1.000

PERCENT_RANK: (rank - 1) / (total_rows - 1). First row = 0.0, last = 1.0
CUME_DIST:    fraction of rows <= current value. First row > 0.0, last = 1.0

=== Flag top-10% transactions by amount within each account ===
 txn_id  account_id  amount  pct_rank size_tier
   1001         101  4200.0     0.000   Top 10%
   1004         101  4200.0     0.000   Top 10%
   10

---
## Common Pitfalls

**1. ROW_NUMBER() ties are non-deterministic without a tiebreaker**
`ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY txn_date)` assigns arbitrary ranks when multiple rows have the same date. The same query may return different rankings across executions. Always add a unique column (e.g. `txn_id`) as a tiebreaker: `ORDER BY txn_date, txn_id`.

**2. Filtering on window function results requires a subquery or CTE**
`WHERE ROW_NUMBER() OVER (...) = 1` raises a syntax error — window functions are evaluated after WHERE. Wrap in a subquery or CTE and filter in the outer query. This is the standard top-N-per-group pattern.

**3. RANK vs DENSE_RANK — choosing the wrong one for business rules**
`RANK()` produces gaps (1, 1, 3) which can be confusing in customer-facing reports ("you ranked 3rd but there is no 2nd"). `DENSE_RANK()` avoids gaps (1, 1, 2). Choose based on whether gaps in the sequence are meaningful. For deduplication (keeping exactly one row per group), use `ROW_NUMBER()` — it always produces unique ranks.

**4. NTILE distributes rows unevenly when the count isn't divisible**
`NTILE(4)` over 23 rows creates buckets of sizes 6, 6, 6, 5 — the first buckets get the extra rows. This is correct behaviour but means buckets aren't perfectly equal. If equal-count buckets are critical, use `PERCENT_RANK()` thresholds instead.

**5. PERCENT_RANK is 0.0 for the first row — not 1/n**
The formula is `(rank - 1) / (count - 1)`, making the very first row always 0.0. This surprises people expecting it to start at `1/n`. `CUME_DIST()` starts above 0 and reaches exactly 1.0 at the last row — choose based on which scale fits your use case.

**6. Using RANK() for deduplication keeps all tied rows**
`WHERE RANK() = 1` keeps all rows sharing the top rank. If three transactions have the same amount, all three pass. For true deduplication (keep exactly one), always use `ROW_NUMBER() = 1`.


---
*sql_methods_library - Samantha McGarrigle*